# Hansen Ch.27 Censoring and Selection

**Chapter 27**（书稿 PDF 约 **p876–878**，习题 **27.1–27.11**）

理论 step-by-step 见 `Hansen_Ch27_Exercises_Solutions.md`。

本 notebook：Tobit（Olsen 参数化）/ CLAD / OLS，复现 **27.9 CHJ**、**27.10 CPS 工资封顶**、**27.11 DDK 测验分删失**。

> **删失 vs 截断 vs 选择（一眼分清）**
> - 删失：边界点还在（$Y=0$ 很多）→ Tobit / CLAD
> - 截断：边界外整行没了 → 截断 MLE / 截断 NLLS
> - 选择：另一方程决定是否观测 $Y$ → Heckman


## 0. 工具函数


In [ ]:
# Hansen Ch.27 — Tobit / CLAD / OLS（详尽注释）
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import norm

ROOT = Path("../..") / "hansen" / "econometrics" / "data"


def ols(y, X, cluster=None):
    """OLS + HC1 或 cluster-robust SE。"""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    n, k = X.shape
    if cluster is None:
        meat = X.T @ (X * (e ** 2)[:, None])
        V = np.linalg.inv(X.T @ X) @ meat @ np.linalg.inv(X.T @ X)
        V *= n / (n - k)  # HC1
    else:
        g = pd.Series(cluster).astype("category").cat.codes.values
        meat = np.zeros((k, k))
        for gg in np.unique(g):
            idx = g == gg
            s = X[idx].T @ e[idx]
            meat += np.outer(s, s)
        bread = np.linalg.inv(X.T @ X)
        ng = len(np.unique(g))
        V = bread @ meat @ bread * (ng / max(ng - 1, 1)) * ((n - 1) / (n - k))
    return b, np.sqrt(np.maximum(np.diag(V), 0))


def tobit_lower(y, X):
    """
    下删失 Tobit：Y = max(Y*, 0)。
    Olsen 参数：gamma = beta/sigma, nu = 1/sigma（全局凹，易优化）。
    """
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    k = X.shape[1]
    b0 = np.linalg.lstsq(X, y, rcond=None)[0]
    s0 = float(np.std(y - X @ b0) + 1.0)

    def nll(th):
        g, nu = th[:k], th[k]
        if nu <= 1e-8:
            return 1e12
        left = y <= 1e-12
        # P(Y*<=0) = Phi(-X'beta/sig) = Phi(-X'g)
        val = -np.sum(norm.logcdf(-(X[left] @ g)))
        yu = y[~left]
        # 密度：nu * phi(y*nu - X'g)
        val -= np.sum(np.log(nu) - 0.5 * np.log(2 * np.pi) - 0.5 * (yu * nu - X[~left] @ g) ** 2)
        return val

    res = minimize(
        nll, np.r_[b0 / s0, 1 / s0], method="L-BFGS-B",
        bounds=[(None, None)] * k + [(1e-4, None)], options={"maxiter": 500},
    )
    nu = res.x[k]
    return res.x[:k] / nu, 1 / nu, -res.fun, res.success


def tobit_upper(y, X, tau):
    """上删失 Tobit：Y = min(Y*, tau)。"""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    k = X.shape[1]
    b0 = np.linalg.lstsq(X, y, rcond=None)[0]
    s0 = float(np.std(y - X @ b0) + 1e-3)

    def nll(th):
        g, nu = th[:k], th[k]
        if nu <= 1e-8:
            return 1e12
        right = y >= tau - 1e-12
        # P(Y* >= tau) = 1 - Phi((tau - X'b)/sig) = sf(tau*nu - X'g)
        val = -np.sum(norm.logsf(tau * nu - X[right] @ g))
        yu = y[~right]
        val -= np.sum(np.log(nu) - 0.5 * np.log(2 * np.pi) - 0.5 * (yu * nu - X[~right] @ g) ** 2)
        return val

    res = minimize(
        nll, np.r_[b0 / s0, 1 / s0], method="L-BFGS-B",
        bounds=[(None, None)] * k + [(1e-4, None)], options={"maxiter": 500},
    )
    nu = res.x[k]
    return res.x[:k] / nu, 1 / nu, -res.fun, res.success


def clad(y, X, left=None, right=None, n_starts=20, seed=0):
    """
    CLAD：min mean |Y - c(X'b)|，c 为对 latent 指数做删失变换。
    目标非全局凸 → 多起点 Powell。
    """
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    k = X.shape[1]
    x0 = np.linalg.lstsq(X, y, rcond=None)[0]

    def crit(b):
        xb = X @ b
        if left is not None:
            xb = np.maximum(xb, left)
        if right is not None:
            xb = np.minimum(xb, right)
        return np.mean(np.abs(y - xb))

    rng = np.random.default_rng(seed)
    starts = [x0, 0.5 * x0, np.zeros(k)]
    for _ in range(n_starts):
        starts.append(x0 + rng.normal(0, 0.15, k) * (np.abs(x0) + 0.1))
    best = None
    for s in starts:
        r = minimize(crit, s, method="Powell", options={"maxiter": 2500, "xtol": 1e-10, "ftol": 1e-12})
        if best is None or r.fun < best.fun:
            best = r
    return best.x, best.fun


def show(name, b, se=None, labels=None):
    labels = labels or [f"b{i}" for i in range(len(b))]
    print(f"--- {name} ---")
    for i, lab in enumerate(labels):
        if se is None:
            print(f"  {lab:12s} {b[i]:10.4f}")
        else:
            print(f"  {lab:12s} {b[i]:10.4f}  ({se[i]:.4f})")

print("helpers ready")


## Exercise 27.9　CHJ2004：`tinkind` 对收入样条

`tinkind`、`income` ÷1000；`Dincome=(income-1)1{income>1}`。


In [ ]:
chj = pd.read_stata(ROOT / "CHJ2004" / "CHJ2004.dta")
tink = chj["tinkind"].to_numpy(float) / 1000.0
inc = chj["income"].to_numpy(float) / 1000.0
Dinc = (inc - 1.0) * (inc > 1.0)
X = np.column_stack([np.ones(len(tink)), inc, Dinc])
labs = ["const", "income", "Dincome"]

print(f"n={len(tink)}, mean(tinkind)={tink.mean():.3f}")
print(f"(b) censoring rate P(tinkind=0) = {(tink == 0).mean():.3f}  (~25% → 删失偏误不可忽视)")

b, se = ols(tink, X)
show("(a) OLS full sample", b, se, labs)

m = tink > 0
b, se = ols(tink[m], X[m])
show("(c) OLS truncated (tinkind>0)", b, se, labs)
print(f"  n_pos = {m.sum()}")

bt, sig, ll, ok = tobit_lower(tink, X)
show(f"(d) Tobit left@0  (sig={sig:.3f}, logL={ll:.1f}, ok={ok})", bt, None, labs)

bc, mad = clad(tink, X, left=0.0)
show(f"(e) CLAD left@0  (MAD={mad:.3f})", bc, None, labs)

print("""
解读：低段斜率约 -1.5（收入↑，实物转移↓）；高段 income+Dincome≈0（平坦）。
丢掉 0 会放大斜率；Tobit/CLAD 校正删失，CLAD 水平更低（右偏）。
""")


## Exercise 27.10　CPS：log 工资上顶于 3.4（≈\$30/h）

子样本 education≥12；$cwage=\min(lwage,3.4)$。


In [ ]:
cps = pd.read_stata(ROOT / "cps09mar" / "cps09mar.dta")
sub = cps.loc[cps["education"] >= 12].copy()
wage = sub["earnings"].to_numpy(float) / (sub["hours"].to_numpy(float) * sub["week"].to_numpy(float))
ok = np.isfinite(wage) & (wage > 0)
lw = np.log(wage[ok])
ed = sub.loc[ok.values if hasattr(ok, "values") else ok, "education"].to_numpy(float)
# fix boolean index
ed = sub["education"].to_numpy(float)[ok]
X = np.column_stack([np.ones(len(lw)), ed, ed ** 2])
labs = ["const", "educ", "educ2"]
cw = np.minimum(lw, 3.4)

print(f"n={len(lw)}, P(lwage>=3.4)={(lw >= 3.4).mean():.3f}")

b, se = ols(lw, X)
show("(a) OLS true lwage", b, se, labs)

b, se = ols(cw, X)
show("(b) OLS on capped cwage", b, se, labs)

m = cw < 3.4 - 1e-12
b, se = ols(cw[m], X[m])
show("(c) OLS omit capped", b, se, labs)
print(f"  n_uncapped = {m.sum()}")

# 用未封顶 OLS 作 Tobit 初值（Olsen 全局凹，终值应不依赖初值；此处加速）
bt, sig, ll, ok_ = tobit_upper(cw, X, 3.4)
show(f"(d) Tobit right@3.4 (sig={sig:.3f}, logL={ll:.1f})", bt, None, labs)

# 习题提示：CLAD 在顶恰好为 3.4 时可能难识别，改用 3.3
cw33 = np.minimum(lw, 3.3)
bc, mad = clad(cw33, X, right=3.3, n_starts=25)
show(f"(e) CLAD right@3.3 (MAD={mad:.3f})", bc, None, labs)

print("""
不知封顶时 (b) 严重扭曲教育回报；(c) 截断不能修复。
Tobit 部分校正（真实工资并非严格 Tobit DGP，故不会等于 (a)）；
CLAD@3.3 的 educ 斜率更接近真 OLS。
""")


## Exercise 27.11　DDK2011：标准化测验分下删失于 0

回归 tracking + percentile + percentile²；SE **按 school 聚类**。


In [ ]:
ddk = pd.read_stata(ROOT / "DDK2011" / "DDK2011.dta")
ts = ddk["totalscore"].to_numpy(float)
ts = (ts - np.nanmean(ts)) / np.nanstd(ts)
tr = ddk["tracking"].to_numpy(float)
perc = ddk["percentile"].to_numpy(float)
school = ddk["schoolid"].to_numpy()
m = np.isfinite(ts) & np.isfinite(tr) & np.isfinite(perc)
ts, tr, perc, school = ts[m], tr[m], perc[m], school[m]
X = np.column_stack([np.ones(len(ts)), tr, perc, perc ** 2])
labs = ["const", "tracking", "percentile", "perc2"]
ct = np.maximum(ts, 0.0)

print(f"n={len(ts)}, P(testscore<0)={(ts < 0).mean():.3f}  (人为下删失后堆在 0)")

b, se = ols(ts, X, cluster=school)
show("(a) OLS true testscore (cluster school)", b, se, labs)

b, se = ols(ct, X, cluster=school)
show("(b) OLS on ctest=max(ts,0)", b, se, labs)

m2 = ct > 0
b, se = ols(ct[m2], X[m2], cluster=school[m2])
show("(c) OLS truncated ctest>0", b, se, labs)
print(f"  n_pos = {m2.sum()}")

print("""
tracking 真效应约 +0.16 sd；删失 OLS 减半，截断更偏。
高删失率（~57%）时绝不能当没看见。
""")
